# HTR CREMMA Medieval — Fine-tuning Kraken sur Kaggle

**Prérequis :**
- Kaggle Secrets : `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY`
- Accelerator : **GPU T4 x2**

**Ordre d'exécution :** 1 → 2 → 3 → 4 → 5 → 6a → 6b → 7

In [ ]:
# Cellule 1 — Connexion S3
from kaggle_secrets import UserSecretsClient
import boto3
import os
from pathlib import Path

secrets = UserSecretsClient()
s3 = boto3.client(
    's3',
    aws_access_key_id=secrets.get_secret('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=secrets.get_secret('AWS_SECRET_ACCESS_KEY'),
    region_name='eu-west-3'
)
print('Connexion S3 OK')

In [ ]:
# Cellule 2 — Installer Kraken
import subprocess, sys, torch

print(f'PyTorch existant : {torch.__version__}')
print(f'CUDA dispo : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU : {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})')
    # T4 = sm_75 (Turing) — fp16 supporté, bf16 NON supporté
    if cap[0] >= 8:
        print('bf16 supporté (Ampere+)')
    else:
        print('fp16 uniquement (pas de bf16 sur cette architecture)')

TORCH_VER = torch.__version__

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kraken',
    'iso639-lang',
    f'torch=={TORCH_VER}',
], check=True)

print('\n--- Test des imports Kraken ---')
test = subprocess.run(
    [sys.executable, '-c',
     'from kraken.lib.xml import XMLPage; '
     'from kraken.train import KrakenTrainer; '
     'print("Imports OK")'],
    capture_output=True, text=True
)
print(test.stdout)
if test.returncode != 0:
    print('ERREUR IMPORT :')
    print(test.stderr[-1500:])

for pkg in ['kraken', 'torch', 'lightning', 'torchmetrics']:
    v = subprocess.run([sys.executable, '-c', f'import importlib.metadata as m; print(m.version("{pkg}"))'],
                       capture_output=True, text=True)
    print(f'{pkg}: {v.stdout.strip() or v.stderr.strip()}')

In [ ]:
# Cellule 3 — Télécharger splits et modèle de base
os.makedirs('/kaggle/working/splits', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)

s3.download_file('htr-cremma-medieval', 'splits/train.txt', '/kaggle/working/splits/train.txt')
s3.download_file('htr-cremma-medieval', 'splits/dev.txt', '/kaggle/working/splits/dev.txt')
s3.download_file('htr-cremma-medieval', 'splits/test.txt', '/kaggle/working/splits/test.txt')
s3.download_file('htr-cremma-medieval', 'base-model/cremma_generic.mlmodel', '/kaggle/working/cremma_generic.mlmodel')
print('Splits et modèle de base OK')

In [ ]:
# Cellule 4 — Convertir les chemins Windows → Kaggle
for nom in ['train.txt', 'dev.txt', 'test.txt']:
    path = f'/kaggle/working/splits/{nom}'
    with open(path) as f:
        lignes = f.readlines()
    nouvelles = []
    for ligne in lignes:
        ligne = ligne.strip().replace('\\', '/')
        # Remplacer le prefix local par le prefix Kaggle (grayscale)
        for prefix in ['preprocessed/', 'preprocessed_grayscale/', 'preprocessed_binary/']:
            idx = ligne.find(prefix)
            if idx != -1:
                nouvelles.append(f'/kaggle/working/preprocessed_grayscale/{ligne[idx + len(prefix):]}\n')
                break
    with open(path, 'w') as f:
        f.writelines(nouvelles)
    print(f'{nom} : {len(nouvelles)} lignes')

with open('/kaggle/working/splits/train.txt') as f:
    print('Exemple :', f.readline().strip())

In [ ]:
# Cellule 5 — Télécharger les données prétraitées depuis S3 (parallèle)
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# preprocessed_grayscale = mode L (niveaux de gris) — compatible Kraken
S3_PREFIX = 'preprocessed_grayscale'

paginator = s3.get_paginator('list_objects_v2')
objets = []
for page in paginator.paginate(Bucket='htr-cremma-medieval', Prefix=f'{S3_PREFIX}/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith('/') or '.' not in key.split('/')[-1]:
            continue
        objets.append(key)

print(f'{len(objets)} fichiers à télécharger depuis s3://htr-cremma-medieval/{S3_PREFIX}/')

def download(key):
    local_path = Path('/kaggle/working') / key
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file('htr-cremma-medieval', key, str(local_path))
    return key

total, erreurs = 0, 0
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(download, k): k for k in objets}
    for fut in as_completed(futures):
        try:
            fut.result()
            total += 1
            if total % 100 == 0:
                print(f'{total}/{len(objets)} fichiers téléchargés...')
        except Exception as e:
            erreurs += 1
            print(f'Erreur : {e}')

print(f'Terminé — {total} fichiers, {erreurs} erreurs')

In [ ]:
# Cellule 6a — Précompiler les données ALTO en binaire Kraken
import subprocess, os, time, datetime

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TERM'] = 'dumb'

for split, src in [('train', '/kaggle/working/splits/train.txt'), ('dev', '/kaggle/working/splits/dev.txt')]:
    out = f'/kaggle/working/splits/{split}.arrow'
    with open(src) as f:
        fichiers = [l.strip() for l in f if l.strip()]
    print(f'\n=== Compilation {split} ({len(fichiers)} fichiers) ===')
    print(f'Début : {datetime.datetime.now().strftime("%H:%M:%S")}')
    t0 = time.time()

    cmd = ['ketos', 'compile', '-f', 'alto', '-o', out] + fichiers
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=0, env=env)
    buf = ''
    while True:
        ch = proc.stdout.read(1)
        if not ch:
            break
        if ch == '\r':
            print('\r' + buf, end='', flush=True)
            buf = ''
        elif ch == '\n':
            print(buf)
            buf = ''
        else:
            buf += ch
    if buf:
        print(buf)
    proc.wait()

    elapsed = time.time() - t0
    if proc.returncode == 0:
        size = os.path.getsize(out) / 1024 / 1024
        print(f'→ {split}.arrow : {size:.1f} MB')
        print(f'→ Durée : {elapsed/60:.1f} min ({elapsed:.0f}s)')
    else:
        print(f'→ ERREUR compilation {split} (code {proc.returncode})')

In [ ]:
# Cellule 6b — Fine-tuning depuis cremma_generic (T4)
import subprocess, os, time, datetime, re

# Manifests pointant vers les .arrow
with open('/kaggle/working/splits/train_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/train.arrow\n')
with open('/kaggle/working/splits/dev_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/dev.arrow\n')

cmd = [
    'ketos',
    '--device', 'cuda:0',
    '--workers', '4',
    '--precision', '16-mixed',
    '--threads', '4',
    'train',
    '-f', 'binary',
    '-t', '/kaggle/working/splits/train_binary.txt',
    '-e', '/kaggle/working/splits/dev_binary.txt',
    '-i', '/kaggle/working/cremma_generic.mlmodel',
    '--resize', 'union',
    '-o', '/kaggle/working/models/htr_cremma_ft',
    '-q', 'early',
    '-N', '50',
    '--lag', '10',
    '--min-epochs', '3',
    '-B', '8',
    '-r', '0.0001',
    '--augment',
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TERM'] = 'dumb'

print('Commande :', ' '.join(cmd))
print(f'Début    : {datetime.datetime.now().strftime("%H:%M:%S")}')
print('─' * 60)

t_global = time.time()
t_epoch = time.time()
epoch_num = 0
best_acc = 0.0
best_epoch = 0

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=0, env=env)
buf = ''
while True:
    ch = proc.stdout.read(1)
    if not ch:
        break
    if ch == '\r':
        print('\r' + buf, end='', flush=True)
        buf = ''
    elif ch == '\n':
        line = buf
        print(line)

        # Détecter début d'epoch
        m = re.search(r'stage\s+(\d+)', line, re.IGNORECASE)
        if m:
            epoch_num = int(m.group(1))
            t_epoch = time.time()

        # Détecter val_accuracy
        m_acc = re.search(r'val_accuracy[:\s]+([\d.]+)', line, re.IGNORECASE)
        if m_acc:
            acc = float(m_acc.group(1))
            elapsed_epoch = time.time() - t_epoch
            elapsed_total = time.time() - t_global
            if acc > best_acc:
                best_acc = acc
                best_epoch = epoch_num
            print(f'  ┌─ Epoch {epoch_num:>2} | val_accuracy={acc:.4f} ({acc*100:.2f}%) | '
                  f'CER={((1-acc)*100):.2f}% | '
                  f'Durée={elapsed_epoch:.0f}s | '
                  f'Total={elapsed_total/60:.1f}min')
            print(f'  └─ Meilleur : epoch {best_epoch} → {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')

        # Détecter patience / early stopping
        m_pat = re.search(r'patience\s+(\d+)/(\d+)', line, re.IGNORECASE)
        if m_pat:
            cur, total = int(m_pat.group(1)), int(m_pat.group(2))
            remaining = total - cur
            print(f'  ⚡ Patience : {cur}/{total} — early stop dans {remaining} epoch(s) sans amélioration')

        buf = ''
    else:
        buf += ch

if buf:
    print(buf)
proc.wait()

elapsed_total = time.time() - t_global
print('─' * 60)
print(f'Fin      : {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'Durée totale : {elapsed_total/60:.1f} min')
print(f'Meilleur résultat : epoch {best_epoch} → {best_acc*100:.2f}% accuracy / {(1-best_acc)*100:.2f}% CER')
print(f'Code retour : {proc.returncode}')

In [ ]:
# Cellule 7 — Uploader le modèle final sur S3
import glob

modeles = glob.glob('/kaggle/working/models/*.mlmodel')
print(f'Modèles trouvés : {modeles}')

for modele in modeles:
    nom = Path(modele).name
    s3.upload_file(modele, 'htr-cremma-medieval', f'output/{nom}')
    print(f'Uploadé : s3://htr-cremma-medieval/output/{nom}')